<br> 

***  

# <center>**Notebook 02 – Experimental Scenarios**</center>

<br>

<center>Supporting materials for</center>

### <center>MSc Data Science</center>
## <center>Module: DSM500 Final Project</center>

<br>

<center>5 October 2026</center>

***
<br>

**Purpose of this notebook**  
This notebook creates scenarios which support the experimental grid of sample size versus imbalance ratio (IR). Scenarios are represented as lists of indices of the training data and saved as plain text files. For each scenario, one complete set and k training and validation folds are created.

```
..
    |-- data
        |-- scenarios
            |-- scenario-<size tier>-ir<IR>-b<bootstrap>
                |-- training-<tier>-ir<IR>-b<bootstrap>.txt
                |-- folds
                    |-- fold-0-train-ids.txt
                    |-- fold-0-eval-ids.txt
                    |-- ...
                    |-- fold-<k>-train-ids.txt
                    |-- fold-<k>-eval-ids.txt
```

## Read the training pool

Save indices in the `training` subfolder within each scenario folder.

In [6]:
import pandas as pd

# Read the training set
X_train = pd.read_parquet('..\\data\\processed\\X_train.parquet', engine='pyarrow')
y_train = pd.read_parquet('..\\data\\processed\\y_train.parquet', engine='pyarrow')

print(X_train.shape, y_train.shape)

(276759, 125) (276759, 1)


In [7]:
print(X_train.head(2))
print(y_train.head(2))

        NAME_CONTRACT_TYPE CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY  \
ID                                                                      
371646                True           F         False             True   
139700                True           F          True             True   

        CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
ID                                                                
371646             0          180000.0    781920.0      28215.0   
139700             1          202500.0   1006920.0      42790.5   

        AMT_GOODS_PRICE  NAME_TYPE_SUITE  ... AMT_REQ_CREDIT_BUREAU_DAY  \
ID                                        ...                             
371646         675000.0    Unaccompanied  ...                       0.0   
139700         900000.0  Spouse, partner  ...                       0.0   

       AMT_REQ_CREDIT_BUREAU_WEEK AMT_REQ_CREDIT_BUREAU_MON  \
ID                                                            
371646     

## Helper functions to create scenario and CV folds

### Scenario creation (`create_scenarios`)

Create scenarios for the specified bootstraps by sampling from thge training pool. Specify a random state based on the sample size, number of minority class samples, and bootstrap number.

In [3]:
import os
import pandas as pd
import numpy as np

def create_scenarios(X:pd.DataFrame, y:pd.DataFrame, tier, tier_size, ir, bootstraps):
    '''
    Create a random, stratified dataset based on sample size and imbalance ratio for each scenario.

    Parameters
    ----------
    X, y : DataFrame, Training pool features and labels
    tier : str, Size tier label
    tier_size : int, number of samles in the tier
    ir : int, imbalance ratio
    bootstraps : list, bootstrap numbers
    '''
    # Calculate majority and minority sizes from IR
    # If the minority set requires approximately all the available minority samples, use them all
    X_maj = X[y['TARGET']==0]
    X_min = X[y['TARGET']==1]

    # Calculate the number of majority and minority samples required. If approximately all the 
    # minority samples are required then use them all.
    n_min = round(tier_size / (1+ir))
    if abs(X_min.shape[0] - n_min) <= 10:
        n_min = X_min.shape[0]
    n_max = tier_size - n_min
    
    print(f'Generating {len(bootstraps)} samples for {tier} size-tier with IR: {ir:.2f} ...')
    for b in bootstraps:
        # Generate random sets of majority and minority indices then concatenate them.
        # Ensure all samples have known, unique random_states for repeatability:
        #   <sample size> + <number of minority samples> + <bootstrap number>
        random_state = tier_size + n_min + b
        max_indices = X_maj.sample(n_max, random_state=random_state).index
        min_indices = X_min.sample(n_min, random_state=random_state).index
        indices = np.concatenate([max_indices, min_indices])
        np.random.shuffle(indices)

        # Save the sample using name like 'large-ir003-b01.txt in the correct subfolder
        # Create the path if it does not exist
        name = f'{tier}-ir{ir:03d}-b{b:02d}'
        full_path = os.path.join('..\\scenarios', name, 'training-'+name+'.txt')
        print('  ', full_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        np.savetxt(full_path, indices, fmt='%d')

In [2]:
# TEST
# Create one balanced sample of 5000 rows to test the pipeline
#create_scenarios(X_train, y_train, 'atest', 5000, 1, [0])

### Fold creation function (`create_cv_folds`)

Using the scenario training sets created in the previous cell, create K stratified folds of training and validation sets for each scenario. The original labels are referenced to allow the correct proportions of majority and minority class to be selected.  

A K of 5 is selected as a reasonable balance between reducing variance amongst folds and having sufficient minority samples in the folds of the smallest most imbalanced scenario. K is held constant to avoid introducing a possible confounding factor into the study.

In [4]:
# import os
from sklearn.model_selection import StratifiedKFold
import numpy as np

def create_cv_folds(tier, ir, k_folds, b):
    '''
    Generate train and validation indices for the scenario
    '''

    # Read the indices for the full training set for the scenario
    scenario_name = f'{tier}-ir{ir:03d}-b{b:02d}'
    fullpath = os.path.join(
        '..\\scenarios', scenario_name, 
        'training-'+scenario_name+'.txt')
    scenario_indices = np.loadtxt(fullpath, dtype=int).tolist()
    
    # Select the scenario from the full train DataFrame
    X_scenario = X_train.loc[scenario_indices]
    y_scenario = y_train.loc[scenario_indices]
    
    # Generate the fold indices. Ensure different random state is used per bootstrap
    random_state = 42 + b
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    folds = list(skf.split(X_scenario, y_scenario))
    
    # Lookup the sample Ids using the fold indices and save them as
    #     fold-k-train-ids.txt
    #     fold-k-eval-ids.txt
    for k, f in enumerate(folds):
        fold_train_ids = (X_scenario.iloc[f[0]].index)
        fold_eval_ids = (X_scenario.iloc[f[1]].index)
    
        # Save training and evaluation folds in the `folds` subfolder within each scenario
        fullpath = os.path.join(
            '..\\scenarios', scenario_name, 'folds',
            'fold-'+str(k)+'-train-ids.txt')
        os.makedirs(os.path.dirname(fullpath), exist_ok=True)
        np.savetxt(fullpath, fold_train_ids, fmt='%d')
        
        fullpath = os.path.join(
            '..\\scenarios', scenario_name, 'folds',
            'fold-'+str(k)+'-eval-ids.txt')
        os.makedirs(os.path.dirname(fullpath), exist_ok=True)  # Should have no effect
        np.savetxt(fullpath, fold_eval_ids, fmt='%d')

In [30]:
#TEST
#create_cv_folds('atest', 1, 5, 0)

## Create the scenarios and folds

Define a dict scenario based on tier, IRs, and bootstraps required, then use them to call the help functions.

### dicts for the various experiments

In [36]:
# Only the tier name, IRs and number of folds are required right now. Store size for later use.
scenarios_a = {
    'large': {'size': 89368, 'irs': [3, 9, 27, 81, 243], 'k_folds': 5},
    'medium': {'size': 44684, 'irs': [1, 3, 9, 27, 81, 243], 'k_folds': 5},
    'small': {'size': 22342, 'irs': [1, 3, 9, 27, 81, 243], 'k_folds': 5}
}

# Experimental grid - small sizes
scenarios_b = {
    'smalla': {'size': 10000, 'irs': [1, 3, 9], 'k_folds': 5},
    'smallb': {'size': 5000, 'irs': [1, 3, 9], 'k_folds': 5},
    'smallc': {'size': 2500, 'irs': [1, 3, 9], 'k_folds': 5},
    'smalld': {'size': 1250, 'irs': [1, 3, 9], 'k_folds': 5}
}
# Expermental grid - extra small sizes
irs = [1, 3, 9, 27, 81]
scenarios_c = {
    'xsmalla': {'size': 800, 'irs': irs, 'k_folds': 5},
    'xsmallb': {'size': 400, 'irs': irs, 'k_folds': 5},
    'xsmallc': {'size': 200, 'irs': irs, 'k_folds': 5},
    'xsmalld': {'size': 100, 'irs': irs, 'k_folds': 5}
}

# Scenario to test the effect of augmenting with real data
scenarios_d = {'smallaug': {'size': 22342, 'irs': [1, 3, 9, 27, 81], 'k_folds': 5}}

# Additional diagnostics - extreme imbalance
scenarios_e = {
    'medium': {'size': 44684, 'irs': [729], 'k_folds': 5}
}

# Use 1 boostrap for all scenarios
bootstraps = [0]

### Call the helper functions: `create_scenarios`, `create_cv_folds`

In [41]:
# Call the scenario and fold creation functions for required scenarios

# Select one of the defined scenario sets
scenarios = scenarios_a

# Generate scenarios for the specified experimental grid
for tier, params in scenarios.items():
    for ir in params['irs']:
        create_scenarios(X_train, y_train,
                         tier=tier, tier_size=params['size'], 
                         ir=ir, 
                         bootstraps=bootstraps)

print('\n -- All scenarios created --\n')

# For each scenario, bootstrap, and fold, generate train and eval indices
for tier, params in scenarios.items():
    k_folds = params['k_folds']
    for ir in params['irs']:
        print(f'Generating {k_folds} folds for {len(bootstraps):d} bootstraps for {tier} '\
              f'tier with IR: {ir:d}: ', end='')
        for b in bootstraps:
            print('.', end='')
            create_cv_folds(tier,
                            ir,
                            k_folds,
                            b)
        print('  Complete')

print('\n -- All folds created --')

Generating 1 samples for large size-tier with IR: 3.00 ...
   ..\scenarios\large-ir003-b00\training-large-ir003-b00.txt
 Complete
Generating 1 samples for large size-tier with IR: 9.00 ...
   ..\scenarios\large-ir009-b00\training-large-ir009-b00.txt
 Complete
Generating 1 samples for large size-tier with IR: 27.00 ...
   ..\scenarios\large-ir027-b00\training-large-ir027-b00.txt
 Complete
Generating 1 samples for large size-tier with IR: 81.00 ...
   ..\scenarios\large-ir081-b00\training-large-ir081-b00.txt
 Complete
Generating 1 samples for large size-tier with IR: 243.00 ...
   ..\scenarios\large-ir243-b00\training-large-ir243-b00.txt
 Complete
Generating 1 samples for medium size-tier with IR: 1.00 ...
   ..\scenarios\medium-ir001-b00\training-medium-ir001-b00.txt
 Complete
Generating 1 samples for medium size-tier with IR: 3.00 ...
   ..\scenarios\medium-ir003-b00\training-medium-ir003-b00.txt
 Complete
Generating 1 samples for medium size-tier with IR: 9.00 ...
   ..\scenarios\medi

## Create real samples for augmentation

One experiment compares the effects of data augmention using synthetic data with real data. To support this, create select augmentation data sets of real data from the training pool. This data will be used instead of CTGAN-generated synthetic data when augmenting the scenario. Because synthetic data is expected not to have missing values, these must be imputed in the real data being used to replace it.

Due to the limited number of minority samples in the training pool, data can not be augmented by more than 200% for the IR 3 scenario.

### Imputer for missing values

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Create a preprocessor to replace missing with median. Only `float64` type columns have missing 
# values.
numeric_cols = X_train.select_dtypes(include=['float64']).columns
hf_preprocess = ColumnTransformer(
    transformers=[('median_imputer', SimpleImputer(strategy='median'), numeric_cols)],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

### Read the training samples (`hf_read_scenario_data`)

In [48]:
import os
from pathlib import Path

def hf_augmentation_data(tier, ir, bootstrap, augs):
    '''
    Get the training data for a scenario, including only minority class samples.
    
    Summary
    -------
    Generate the path to the indices from the input parameters. Read the indices then select
    the training data from the full training set, impute missing values then return the positive 
    samples.
    '''
    # Do not augment IR=1
    if ir==1:
        print('Message: Skipping IR 1')
        return
        
    # Get the path to the indices
    scenario = f'{tier}-ir{ir:03d}-b{bootstrap:02}'
    root = Path('..') / 'scenarios' / scenario

    # Read the indices for the scenario
    train_file = f'training-{scenario}.txt'
    train_path = root / train_file
    train_indices = np.loadtxt(train_path, dtype=int)
    n_samples = len(train_indices)
    n_pos = int(n_samples / (ir + 1))

    # Create a dataset which excludes the training set and use this to augment it
    X_augmentation = X_train.drop(train_indices)
    y_augmentation = y_train.drop(train_indices)
    X_pos = X_augmentation[y_augmentation['TARGET']==1]

    # Train an imputer to impute the augmentation samples
    hf_preprocess.fit(X_pos)

    # Create an augmentation dataset for each aug
    print(f'Creating augmentation data for {tier} IR {ir} augs: ', end='')
    
    for aug in augs:
        # Skip 0% augmentation
        if aug==0:
            continue

        print(f'{aug} ', end='')
        # Create unique random states for each ir-bootstrap-aug combination
        random_state = (ir * 100000) + (bootstrap * 1000) + aug
        n_aug = int(n_pos * aug/100)

        # Select a sample and input the missing values, then save is as a parquet file
        real_aug = X_pos.sample(n_aug, random_state=random_state)
        real_aug_imp = hf_preprocess.transform(real_aug)

        # Save the real data to the synthetic folder
        path = f'..\\scenarios\\{tier}-ir{ir:03d}-b{bootstrap:02}\\synthetic\\'
        file = f'synthetic-aug{aug:03d}.parquet'
        os.makedirs(path, exist_ok=True)
        real_aug.to_parquet(path+file, engine='pyarrow')
    print('Complete')

### Call the helper function to create real augmentation samples

In [49]:
# Call the helper function to create required augmentation samples for small IR 1
# NOTE Insufficient samples for 400% augmentation the IR 3 scenario.
tiers = ['smallaug']
irs = [1, 9, 27, 81]
augs = [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200, 400]
augs_ir3 = [0, 4, 8, 13, 18, 22, 28, 33, 38, 44, 50, 100, 200]

# Scenarios with IRs: 9, 27, and 81
for tier in tiers:
    for ir in irs:
        hf_augmentation_data(tier, ir, 0, augs)

# Scenario with IRs: 3
for tier in tiers:
    for ir in [3]:
        hf_augmentation_data(tier, ir, 0, augs_ir3)

print('\n-- Processing complete --')

Message: Skipping IR 1
Creating augmentation data for smallaug IR 9 augs: 4 8 13 18 22 28 33 38 44 50 100 200 400 Complete
Creating augmentation data for smallaug IR 27 augs: 4 8 13 18 22 28 33 38 44 50 100 200 400 Complete
Creating augmentation data for smallaug IR 81 augs: 4 8 13 18 22 28 33 38 44 50 100 200 400 Complete
Creating augmentation data for smallaug IR 3 augs: 4 8 13 18 22 28 33 38 44 50 100 200 Complete

-- Processing complete --


*<center>-- End of notebook --</center>*